# Lesson 5: Layers and regions

**You will learn:** how `region_layer_map` links brain regions to model layers, and how to record several regions in one pass.
**Requires:** CPU only, no downloads, under a minute. Builds on [Lesson 3](03_wrap_a_pytorch_model.ipynb).

A neural benchmark asks for activity from a brain region, such as V1 or IT. The model does not have regions; it has layers. `region_layer_map` states which layer you propose as the model's counterpart of each region. This mapping is a hypothesis: a benchmark measures how well it holds.

In [1]:
import warnings
warnings.filterwarnings('ignore')  # keep the lesson output short

In [2]:
import tempfile, pathlib
import numpy as np
from PIL import Image
from brainscore_core.supported_data_standards.brainio.stimuli import StimulusSet

# Eight small synthetic images: four bright, four dark. They stand in for real stimuli.
folder = pathlib.Path(tempfile.mkdtemp())
rng = np.random.default_rng(0)
rows = []
for i in range(8):
    bright = i % 2 == 0
    low, high = (150, 255) if bright else (0, 100)
    path = folder / f'image_{i}.png'
    Image.fromarray(rng.integers(low, high, (16, 16, 3), dtype=np.uint8)).save(path)
    rows.append({'stimulus_id': f'image_{i}', 'image_path': str(path),
                 'label': 'bright' if bright else 'dark'})

stimuli = StimulusSet(rows)
stimuli.identifier = 'toy-bright-dark'
stimuli.stimulus_paths = dict(zip(stimuli['stimulus_id'], stimuli['image_path']))
stimuli

,stimulus_id,image_path,label
0,image_0,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
1,image_1,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark
2,image_2,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
3,image_3,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark
4,image_4,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
5,image_5,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark
6,image_6,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,bright
7,image_7,/var/folders/h6/g53mxg516n538136hh74630r0000gn...,dark


In [3]:
import torch

torch.manual_seed(0)
# A tiny, untrained convolutional network. Its weights are random.
net = torch.nn.Sequential(
    torch.nn.Conv2d(3, 4, kernel_size=3),  # layer '0'
    torch.nn.ReLU(),                       # layer '1'
    torch.nn.AdaptiveAvgPool2d(2),         # layer '2'
    torch.nn.Flatten(),                    # layer '3'
    torch.nn.Linear(16, 8),                # layer '4'
).eval()

def preprocess(paths):
    """Load image files into a (batch, 3, height, width) float array."""
    images = [np.asarray(Image.open(p).convert('RGB'), dtype=np.float32) / 255 for p in paths]
    return np.stack(images).transpose(0, 3, 1, 2)

In [4]:
from brainscore_core import BrainScoreModel
from brainscore.model_helpers.vision_wrapper import VisionWrapper

model = BrainScoreModel(
    identifier='toy-cnn',
    model=net,
    extractor={'vision': VisionWrapper(identifier='toy-cnn', model=net, preprocessing=preprocess)},
    region_layer_map={'V1': '0', 'V4': '2', 'IT': '4'},
)
model.out_channels

{'neural:IT', 'neural:V1', 'neural:V4'}

## Record several regions at once

Request several `neural:` channels in one session. The network runs once, and the result is a dictionary keyed by channel.

In [5]:
from brainscore_core.streaming import BufferedSession

session = BufferedSession(stimulus_set=stimuli,
                          requested_output_channels=['neural:V1', 'neural:V4', 'neural:IT'])
model.participate(session)
recordings = session.collect()
model.reset()
{channel: assembly.shape for channel, assembly in recordings.items()}

activations:   0%|          | 0/64 [00:00<?, ?it/s]

layer packaging:   0%|          | 0/3 [00:00<?, ?it/s]

{'neural:V1': (8, 784), 'neural:V4': (8, 16), 'neural:IT': (8, 8)}

V1 maps to the first convolution, which keeps every spatial position of every channel, so it has many more units than IT, which maps to the 8 outputs of the last layer.

## Compare the regions

Each region's activity is a matrix of stimuli by units. A simple comparison is to ask, for each region, how differently it responds to bright and dark images.

In [6]:
for channel, assembly in recordings.items():
    values = assembly.values
    bright = values[stimuli['label'] == 'bright'].mean(axis=0)
    dark = values[stimuli['label'] == 'dark'].mean(axis=0)
    print(f'{channel:>10}: mean |bright - dark| = {np.abs(bright - dark).mean():.3f}')

 neural:V1: mean |bright - dark| = 0.163
 neural:V4: mean |bright - dark| = 0.029
 neural:IT: mean |bright - dark| = 0.020


## Routing regions in multi-tower models

Some models have separate towers for different inputs, such as an image encoder and a text encoder. Then the same layer name can exist in both towers. Use a `LayerSelector` with a `modality` to say which tower a region reads from:

```python
from brainscore_core import LayerSelector

region_layer_map = {
    'IT': LayerSelector(name='encoder.layers.10', modality='vision'),
    'language_system': LayerSelector(name='encoder.layers.10', modality='text'),
}
```

A plain string, as used above, is shorthand for `LayerSelector(name=...)`. A region drawn from units in several layers uses `CompositeSelector`, covered in [Lesson 8](08_choose_units.ipynb).

## Summary

- `region_layer_map` is your hypothesis about which layer corresponds to which region.
- Request several `neural:<region>` channels to record them in one forward pass.
- Use `LayerSelector(..., modality=...)` when a model has several input towers.

**Next:** [Lesson 6](06_behavior.ipynb) collects behavior instead of activity.